# Emoji wrap boundary

Timestamp: 2026-09-11 23:58:22 +04


## Hypothesis

Replacing only the ordinary space after the final exclamation mark with the JavaScript escape `\u00A0` preserves the visible wording while preventing a line break between the punctuation and 🥳.


## Method

Inspect the assignment source, execute the inline JavaScript in Node with a mocked DOM and deterministic random value selecting Burrito, assert character code 160 immediately before 🥳 and normalized display text, verify 20 dishes and a different consecutive pick, then run `node --check` on the extracted inline script.


In [1]:
from pathlib import Path
html = Path("../index.html").read_text()
expected = r"result.textContent = `Today's pick: ${choice.name}!\u00A0🥳`;"
assert html.count(expected) == 1
assert "result.textContent" in expected and "innerHTML" not in expected
print("SOURCE_ESCAPE_CHECK PASS")
print(expected)


SOURCE_ESCAPE_CHECK PASS
result.textContent = `Today's pick: ${choice.name}!\u00A0🥳`;


In [2]:
import subprocess
node_check = r"""const fs=require("fs"),vm=require("vm"),assert=require("assert");
const html=fs.readFileSync("../index.html","utf8");
const scripts=[...html.matchAll(/<script>([\s\S]*?)<\/script>/g)];
assert.strictEqual(scripts.length,1);
let click;
const make=()=>({classList:{add(){},remove(){}},hidden:false,offsetWidth:0});
const elements={"#generate-button":{addEventListener(type,fn){if(type==="click") click=fn;}},
  "#result":make(),"#food-photo":make(),"#photo-placeholder":make(),"#photo-credit":make()};
const mockMath=Object.create(Math); mockMath.random=()=>9/20;
const context={document:{querySelector:s=>elements[s]},Math:mockMath};
vm.createContext(context);
vm.runInContext(scripts[0][1]+"\nglobalThis.__lunchOptions=lunchOptions;",context);
assert.strictEqual(context.__lunchOptions.length,20);
const picks=[];
for(let i=0;i<40;i++){click(); picks.push(elements["#result"].textContent);}
const first=picks[0];
const emoji=first.indexOf("🥳");
assert(emoji>0);
assert.strictEqual(first.charCodeAt(emoji-1),160);
assert.notStrictEqual(first[emoji-1]," ");
assert.strictEqual(first.replace(/\u00a0/g," "),"Today's pick: Burrito! 🥳");
assert(picks.every((pick,i)=>i===0||pick!==picks[i-1]));
console.log("JS_MOCK_CHECK PASS");
console.log(`normalized_display=${first.replace(/\u00a0/g," ")}`);
console.log(`code_point_before_emoji=${first.charCodeAt(emoji-1)}`);
console.log(`ordinary_space_before_emoji=${first.charCodeAt(emoji-1)===32}`);
console.log(`choices=${context.__lunchOptions.length}`);
console.log(`clicks=${picks.length} consecutive_duplicates=0`);"""
completed = subprocess.run(["node", "-e", node_check], check=True, text=True, capture_output=True)
print(completed.stdout, end="")


JS_MOCK_CHECK PASS
normalized_display=Today's pick: Burrito! 🥳
code_point_before_emoji=160
ordinary_space_before_emoji=false
choices=20
clicks=40 consecutive_duplicates=0


In [3]:
import subprocess
from pathlib import Path
import re
html = Path("../index.html").read_text()
scripts = re.findall(r"<script>([\s\S]*?)</script>", html)
assert len(scripts) == 1
subprocess.run(["node", "--check"], input=scripts[0], check=True, text=True, capture_output=True)
print("INLINE_JS_SYNTAX PASS")


INLINE_JS_SYNTAX PASS


## Interpretation

The source retains `textContent` and contains the requested `\u00A0` escape. Runtime output places character code 160 immediately before 🥳, contains no ordinary space at that boundary, and normalizes to `Today's pick: Burrito! 🥳` for display. The mocked run retained 20 dishes and produced different consecutive choices. Inline JavaScript syntax passed.
